# The Durian Notebook - decision models vs LLMs on 83 durian opinions

Three durians - **Musang King**, **Black Thorn** and **Red Prawn** - and 83 real opinions about them
in English, Chinese and Malay, quoted from blogs, forums and news. Every system reads each opinion
once and answers the same 18 questions about it. **Part 1** measures accuracy, time and money for
*decision models* (Jev, OpenAI's Decisions API, Kev and other local models) against ordinary LLMs.
**Part 2** builds a Durian Elo from the answers and asks whether the cheap models crown the same
king as the people did.

This notebook is a tutorial: each section of *Meet the solutions* shows, in plain sight, how to call
one API on one opinion. The machinery - the batch runner, the `.cache/` cache, metrics, costs, the
Elo, the charts and the README writer - lives in `src/` (`benchmarking.py`, `metrics.py`, `elo.py`,
`charts.py`, `report.py`; the questions and payload builders are in `schema.py`). Facts about the
models are as of October 2026; vendor numbers are vendor numbers.

In [ ]:
SMOKE = True  # True: 5 opinions covering all three durians and all three languages; False: all 83
FORCE_RERUN = False  # True: ignore .cache/ and call every API again
RUN_ONLY: list[str] | None = None  # e.g. ["jev", "kev"] to run a subset; None = every available one
BUDGET_USD = 10.0  # the runner stops when the new calls of one run cost more than this

In [ ]:
import json
import os

import pandas as pd

import benchmarking
import charts
import elo
import metrics
import report
from schema import QUESTIONS, QUESTIONS_DECISIONS, QUESTIONS_SYSTEMONE, ROOT, DurianAnswer

pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 90)
os.environ.setdefault("PYDANTIC_AI_NO_BANNER", "1")  # keeps the bonus section's output quiet

solutions = benchmarking.registry()
availability = benchmarking.availability(solutions)


def ready(solution_id: str) -> bool:
    """True when the solution can run now; otherwise print why it is skipped."""
    if availability.loc[solution_id, "status"] == "available":
        return True
    print(f"skipped {solution_id}: {availability.loc[solution_id, 'reason']}")
    return False


availability

`benchmarking.registry()` lists every solution; the table says which ones can run on this machine
right now - a key in `.env` (loaded with python-dotenv, never printed), a reachable Ollama of the
right version, a pulled model. A skipped solution stays skipped: nothing is faked or imputed for it.

## 1. The data

`data/durian_opinions.csv` holds 83 short, verbatim opinions with their sources: 70 in the main set
and 13 reserves from the same pages. The model only ever sees the `quote` column, in its original
language; `quote_en` is a translation for us. The gold labels (which aspects are mentioned, the
overall sentiment, whether the text compares durians and which one it prefers) were drafted by
Claude and can be fixed in the CSV - every model answer is cached, so a fix only re-runs the
evaluation.

In [ ]:
all_opinions = benchmarking.load_opinions()
opinions = benchmarking.smoke_subset(all_opinions) if SMOKE else all_opinions
mode = "SMOKE" if SMOKE else "FULL"
print(f"{len(all_opinions)} opinions in the dataset, {len(opinions)} in this {mode} run")
pd.crosstab(all_opinions["variety"], all_opinions["language"], margins=True)

In [ ]:
for language in ("en", "zh", "ms"):
    example = all_opinions[all_opinions["language"] == language].iloc[0]
    print(f"[{example.id}] {example.quote}")
    if example.quote_en:
        print(f"   -> {example.quote_en}")
    print(f"   {example.source_site} · gold sentiment: {example.gold_overall_sentiment}\n")

confidence = all_opinions["gold_confidence"].value_counts()
print("draft-label confidence:", confidence.to_dict())

## 2. The questions

`config/durian.yaml` is the single source of truth: six aspects (smell, taste, texture, value,
quality, aftermath) give one *mentioned* yes/no question and one *sentiment* score each, and six
more questions follow - 18 per opinion, all sent in **one** request. Three types: yes/no (System One
calls it `noul`, OpenAI `predicate`), pick one (`choice`) and a position on an ordered scale
(`score`). Scores are normalised to 0..1 as the probability-weighted level index divided by the
number of levels minus one.

In [ ]:
questions_table = pd.DataFrame(
    {
        "question": question.name,
        "type": question.kind,
        "gold": question.gold,
        "levels / options": len(question.levels) or len(question.options),
        "instructions": question.instructions,
    }
    for question in QUESTIONS
).set_index("question")
questions_table

`schema.py` turns the questions into the three payload shapes the APIs expect. Below is one
question of each type in each shape - the full payloads carry all 18.

In [ ]:
print("System One (Jev, Kev, the other Ollama decision models): a dict keyed by question name\n")
for name in ("smell_mentioned", "overall_sentiment", "humour"):
    print(json.dumps({name: QUESTIONS_SYSTEMONE[name]}, indent=2, ensure_ascii=False))

In [ ]:
print("OpenAI Decisions API: a list of questions with a name each\n")
for question in QUESTIONS_DECISIONS:
    if question["name"] in ("smell_mentioned", "overall_sentiment", "humour"):
        print(json.dumps(question, indent=2, ensure_ascii=False))

In [ ]:
print("LLMs: a pydantic model (DurianAnswer), passed to the API as the response format\n")
answer_schema = DurianAnswer.model_json_schema()
for name in ("smell_mentioned", "overall_sentiment", "humour"):
    print(json.dumps({name: answer_schema["properties"][name]}, indent=2, ensure_ascii=False))
print("\nsystem prompt for the LLMs (prompts/durian_llm.md):\n")
print(benchmarking.SYSTEM_PROMPT)

## 3. Meet the solutions

A **decision model** (TypeSafe calls it a *System One model*) reads a text - the `state` - and
answers typed questions about it with **probabilities instead of generated text**. Under the hood
the open ones are ordinary LLMs (mostly Qwen 3.5) that only do the reading step: one forward pass
over the input, then a small head scores the options you supplied. Nothing is generated, so there
are no output tokens to bill and an answer comes back in a tenth to half a second. They don't
decide: they estimate and give a confidence, and your code decides what to accept and what to send
for review. Known weak spots, from Jev's own docs: counting, arithmetic, dates, a lean toward the
first option, irony.

Every cell below runs the same opinion through one API and prints the raw response and the
normalised answer. The batch runner in `src/benchmarking.py` makes exactly these calls for every
opinion.

In [ ]:
example = opinions.iloc[0]
print(f"[{example.id}] {example.quote}")
if example.quote_en:
    print(f"   -> {example.quote_en}")

### 3.1 Jev (TypeSafe AI) - the original, in the cloud

Jev launched on 15 September 2026; `jev-latest` is Jev 1.14 as of October 2026. TypeSafe says it is
"not an LLM and not autoregressive, but transformer-based" and has published nothing more. One
request: `POST https://api.typesafe.ai/v1/systemone` with a Bearer key, the `state` and a
`questions` dict. Billing: USD 0.042 per 1M input tokens; the response reports output tokens too,
but TypeSafe does not charge for them.

In [ ]:
import requests

if ready("jev"):
    payload = {"model": "jev-latest", "state": example.quote, "questions": QUESTIONS_SYSTEMONE}
    headers = {"Authorization": f"Bearer {os.environ['TYPESAFE_API_KEY']}"}  # the key stays in .env
    response = requests.post(
        "https://api.typesafe.ai/v1/systemone", headers=headers, json=payload, timeout=60
    )
    response.raise_for_status()
    raw = response.json()
    print(json.dumps(raw, indent=2, ensure_ascii=False))  # every answer carries its probabilities
    print("\nusage:", raw["usage"], "<- output tokens are counted but free")
    print("\nnormalised:", benchmarking.parse_systemone(raw))

TypeSafe's own SDK (`typesafe-sdk`) wraps the same call with typed answers and retries; it takes
our question dicts as they are. With `base_url="http://localhost:11434"` the same client talks to
Kev and the other decision models on your laptop - one request shape for the cloud and for Ollama.

In [ ]:
if ready("jev"):
    from typesafe_sdk import TypeSafeClient

    with TypeSafeClient(timeout=60) as client:  # reads TYPESAFE_API_KEY from the environment
        result = client.system_one(
            state=example.quote, questions=QUESTIONS_SYSTEMONE, model="jev-latest"
        )
    print(result.choices["overall_sentiment"])
    print(result.scores["overall_rating"])
    print(result.usage)

### 3.2 Kev-4B - an open decision model on your laptop, through Ollama

Kev (Jared Palmer, Apache-2.0) is a Qwen 3.5 backbone with a LoRA and a small "pointer head" that
scores the supplied options. Since Ollama 0.35 the local server exposes `/v1/systemone`, the same
request shape as Jev's (up to 64 questions and 64 KiB per request); Kev itself needs Ollama 0.40.4.
No key, no bill - hardware and electricity are yours.

In [ ]:
if ready("kev"):
    payload = {"model": "kev:4b-q8_0", "state": example.quote, "questions": QUESTIONS_SYSTEMONE}
    response = requests.post(f"{benchmarking.OLLAMA_HOST}/v1/systemone", json=payload, timeout=120)
    response.raise_for_status()
    raw = response.json()
    print(json.dumps(raw, indent=2, ensure_ascii=False))
    print("\nnormalised:", benchmarking.parse_systemone(raw))

The Ollama Python library wraps the same endpoint; the questions dict is identical.

In [ ]:
if ready("kev"):
    import ollama

    result = ollama.systemone(
        model="kev:4b-q8_0", state=example.quote, questions=QUESTIONS_SYSTEMONE
    )
    print(result.answers["overall_sentiment"])
    print(result.usage)

### 3.3 Clef Flash, Laya and the other local decision models - same call

Every decision model in Ollama's library answers the same `/v1/systemone` request. **Clef Flash**
is Cloudflare's 9B model, a Qwen3.5-9B fine-tune that also takes images (Ollama 0.35.1 or later).
**Laya** (Convai Innovations) is a 421M ModernBERT encoder - the encoder-only approach: English,
512 tokens per question shared between the instructions, the options and the text, so expect it to
struggle with Chinese and Malay. Liquid AI's **d1** (3B, text and images), Together AI's **tev1**
(Qwen3.5-4B, ~2,000-token context) and Bespoke Labs' **Nimble** (9B) take the same call. Only the
pulled ones run.

In [ ]:
for solution_id, model in [
    ("ollama_clef_flash", "clef-flash"),
    ("ollama_laya", "laya"),
    ("ollama_d1", "d1"),
    ("ollama_tev1", "tev1:4b"),
    ("ollama_nimble", "nimble"),
]:
    if ready(solution_id):
        payload = {"model": model, "state": example.quote, "questions": QUESTIONS_SYSTEMONE}
        response = requests.post(
            f"{benchmarking.OLLAMA_HOST}/v1/systemone", json=payload, timeout=120
        )
        response.raise_for_status()
        raw = response.json()
        print(f"\n{model}: {json.dumps(raw['answers']['overall_sentiment'], ensure_ascii=False)}")
        print(f"{model}: usage {raw['usage']}")
        print(f"{model}: {benchmarking.parse_systemone(raw)}")

### 3.4 OpenAI Decisions API - the same idea on GPT-6 Luna

Announced at DevDay on 29 September 2026 and generally available since 22 October: `POST
/v1/decisions`, `client.decisions.create(...)`, only on `gpt-6-luna`. Question types `predicate`,
`choice` and `score` (2 to 10 levels), up to 400 questions per request. USD 0.10 per 1M input
tokens; `output_tokens` comes back as 0 - nothing is generated. Because it runs the same model as the GPT-6 Luna LLM below, the
comparison isolates what the decision interface itself adds.

In [ ]:
from openai import OpenAI

if ready("openai_decisions"):
    client = OpenAI()  # reads OPENAI_API_KEY from the environment
    decision = client.decisions.create(
        model="gpt-6-luna", input=example.quote, questions=QUESTIONS_DECISIONS
    )
    print(decision.model_dump_json(indent=2))
    print("\nnormalised:", benchmarking.parse_openai_decisions(decision.model_dump(mode="json")))

### 3.5 GPT-6 Luna and GPT-5.6 Terra - ordinary LLMs with structured output

The LLM baselines get the same 18 questions as a pydantic model: the Responses API fills it in
(`text_format=DurianAnswer`), so the answer is validated JSON, not free text. GPT-6 Luna runs with
reasoning effort `none` (USD 0.10 / 0.50 per 1M input / output tokens); GPT-5.6 Terra with effort
`low` (USD 2 / 12 per 1M - an aggregator price, to confirm). Here the output tokens are what you pay
for.

In [ ]:
for solution_id, model, effort in [
    ("gpt_6_luna", "gpt-6-luna", "none"),
    ("gpt_5_6_terra", "gpt-5.6-terra", "low"),
]:
    if ready(solution_id):
        client = OpenAI()
        response = client.responses.parse(
            model=model,
            input=[
                {"role": "system", "content": benchmarking.SYSTEM_PROMPT},
                {"role": "user", "content": example.quote},
            ],
            text_format=DurianAnswer,
            reasoning={"effort": effort},
        )
        print(f"\n{model}: {response.output_text}")  # the raw JSON text the model wrote
        print(
            f"{model}: usage {response.usage.input_tokens} in / {response.usage.output_tokens} out"
        )
        print(f"{model}: {benchmarking.parse_llm({'answer': response.output_parsed.model_dump()})}")

### 3.6 A local LLM through Ollama - the free baseline

An ordinary open LLM with structured output: `ollama.chat` takes the JSON schema of `DurianAnswer`
as `format` and temperature 0. The registry picks the first pulled tag from Qwen 3.5, Gemma 4 or
Qwen 3 (`benchmarking.LOCAL_LLM_PREFERENCES`); `think=False` switches off the reasoning phase of
the Qwen models, otherwise every opinion would take minutes.

In [ ]:
if ready("ollama_llm"):
    import ollama

    LOCAL_LLM = benchmarking.pick_local_llm()
    response = ollama.chat(
        model=LOCAL_LLM,
        messages=[
            {"role": "system", "content": benchmarking.SYSTEM_PROMPT},
            {"role": "user", "content": example.quote},
        ],
        format=DurianAnswer.model_json_schema(),
        options={"temperature": 0},
        think=False if benchmarking.ollama_supports_thinking(LOCAL_LLM) else None,
    )
    print(LOCAL_LLM, "->", response.message.content)  # the raw JSON text
    print("tokens:", response.prompt_eval_count, "in /", response.eval_count, "out")
    answer = DurianAnswer.model_validate_json(response.message.content)
    print("\nnormalised:", benchmarking.parse_llm({"answer": answer.model_dump()}))

### 3.7 Optional reference: Claude Opus 5.5 at high effort

The score questions (aspect sentiment, overall rating, humour) have no human labels. A bigger LLM
serves as a reference for them, when `ANTHROPIC_API_KEY` is set: the Messages API parses the same
`DurianAnswer` model (`output_format=`), with `output_config={"effort": "high"}`. USD 4 / 20 per 1M
tokens - the most expensive call in the notebook.

In [ ]:
if ready("reference"):
    from anthropic import Anthropic

    client = Anthropic()  # reads ANTHROPIC_API_KEY
    message = client.messages.parse(
        model="claude-opus-5-5",
        max_tokens=16000,
        system=benchmarking.SYSTEM_PROMPT,
        messages=[{"role": "user", "content": example.quote}],
        output_format=DurianAnswer,
        output_config={"effort": "high"},
    )
    print(message.parsed_output)
    print("usage:", message.usage.input_tokens, "in /", message.usage.output_tokens, "out")
    print("\nnormalised:", benchmarking.parse_llm({"answer": message.parsed_output.model_dump()}))

## 4. Run the battle

One line runs every available solution on every opinion, sequentially, through the cache: one JSON
per call under `.cache/<solution>/<model>/<schema version>/<opinion id>.json` with the request, the
raw response, the latency, the usage and a timestamp. Re-runs read the cache; local models get one
untimed warm-up call; 429 / 529 / 5xx answers are retried with backoff; a solution that fails three
times in a row is stopped and its rows carry the error. Timing is wall-clock per request with
`time.perf_counter()` - cloud calls include the round trip from Singapore, local ones don't.

In [ ]:
to_run = [s for s in benchmarking.available(solutions) if RUN_ONLY is None or s.id in RUN_ONLY]
print("running:", [s.id for s in to_run])
results = benchmarking.run_benchmark(
    opinions, to_run, force_rerun=FORCE_RERUN, budget_usd=BUDGET_USD
)
n_cached, n_errors = int(results["cached"].sum()), int(results["error"].notna().sum())
print(f"{len(results)} answers · {n_cached} from the cache · {n_errors} errors")
print(
    f"USD {results['cost_usd'].sum():.4f} spent on this run at list prices (cached calls included)"
)
results.loc[results["error"].notna(), ["solution", "opinion_id", "error"]]

## 5. Part 1: who reads durians best

Accuracy against the gold labels: micro-F1 over the six *mentioned* questions (threshold 0.5),
overall-sentiment accuracy, *compares* F1, *preferred* accuracy where the gold says the text
compares, and *variety* accuracy where the quote names its own durian. The **headline** is the
mean of aspects F1, overall-sentiment accuracy, compares F1 and variety accuracy. Time is seconds
per opinion (p50 and p95). Money is the tokens in each response's `usage` times the list prices in
`config/prices.yaml`, for this run and extrapolated to 1M opinions; local models are USD 0 (hardware
not counted).

In [ ]:
summary = metrics.summary_table(results, opinions, benchmarking.solutions_frame(to_run))
part1_columns = [
    "label", "family", "n_answered", "headline", "aspects_f1", "overall_sentiment_acc",
    "compares_f1", "preferred_acc", "variety_acc", "p50_s", "p95_s", "usd_run",
    "usd_per_1m_opinions",
]  # fmt: skip
summary[part1_columns].sort_values("headline", ascending=False).round(3)

In [ ]:
summary[["label", "headline_en", "headline_zh", "headline_ms"]].round(3)

Decision models return probabilities, so every answer carries a confidence: the probability of the
option it chose (for yes/no the larger of p and 1 - p). Binning the gold-scored answers by that
confidence shows how accuracy rises with it, and the cutoff table says what you would keep if you
accepted everything at or above a threshold and sent the rest to a person. LLMs give no probability
(every structured answer would sit in the top bin), so they are left out - you can't route on them.

In [ ]:
solution_info = benchmarking.solutions_frame(to_run)
decision_models = solution_info[solution_info["family"].str.startswith("decision")]
confidences = metrics.answer_confidences(
    results[results["solution"].isin(decision_models.index)], opinions, decision_models["label"]
)
bin_accuracy, bin_counts = metrics.confidence_matrix(confidences)
confidence_table = metrics.annotate(bin_accuracy, bin_counts)
confidence_table

In [ ]:
cutoff_accuracy, cutoff_share = metrics.cutoff_table(confidences)
cutoff_display = metrics.annotate(cutoff_accuracy, cutoff_share, "{:.0%}")
cutoff_display

As a check, the same table on the 70 main-set opinions only (the 13 reserves came from the same
pages) - full runs only.

In [ ]:
main_only = None
if not SMOKE:
    main_only = metrics.summary_table(
        results, opinions[opinions["set"] == "main"], benchmarking.solutions_frame(to_run)
    )
    display(main_only[["label", "n_answered", "headline"]].round(3))

The score questions have no gold. The overall rating is compared with the eight writers who gave a
score themselves (shown as points - no statistics on n = 8), and the sentiment and humour scores
are compared across solutions (Spearman agreement; the reference model is one of the columns when
it ran).

In [ ]:
solution_labels = benchmarking.solutions_frame(to_run)[["label", "family"]]
points = metrics.rating_points(results, opinions).merge(
    solution_labels, left_on="solution", right_index=True
)
points.pivot(index="opinion_id", columns="label", values="overall_rating").join(
    all_opinions.set_index("id")["real_score"]
).round(2)

In [ ]:
if len(to_run) >= 2:
    print("humour - Spearman agreement between solutions")
    display(metrics.agreement_matrix(results, "humour").round(2))
    print("taste sentiment - Spearman agreement between solutions")
    display(metrics.agreement_matrix(results, "taste_sentiment").round(2))

The charts are Plotly figures: interactive here (hover a point for the solution or the opinion), and
saved as 1600 px PNGs by kaleido to `assets/charts/` (a smoke run writes them under
`results/smoke/charts/`). Chart 7 trades money for time: seconds per opinion for every solution,
with the cloud round trip included and the local machine named in the title.

In [ ]:
run_date = report.now_singapore()[:10]
machine = benchmarking.machine_description()
chart_dir = ROOT / "results" / "smoke" / "charts" if SMOKE else ROOT / "assets" / "charts"
figures = [
    charts.cost_vs_accuracy(summary, chart_dir / "01_cost_vs_accuracy.png", run_date),
    charts.time_vs_accuracy(summary, chart_dir / "07_time_vs_accuracy.png", run_date, machine),
    charts.time_per_opinion(summary, chart_dir / "02_time_per_opinion.png", run_date),
    charts.accuracy_by_question(
        metrics.by_question_table(summary), chart_dir / "03_accuracy_by_question.png", run_date
    ),
    charts.accuracy_by_language(
        metrics.by_language_table(summary), chart_dir / "04_accuracy_by_language.png", run_date
    ),
    charts.rating_vs_real(points, chart_dir / "05_rating_vs_real.png", run_date),
]
for figure in figures:
    figure.show()

## 6. Part 2: the Durian Elo

All three durians start at 1500 and play three kinds of matches in date order: **head-to-head**
(K = 32) when a quote prefers one durian over another one it names; **same writer** (K = 24) when
one person reviewed two durians - the higher mean overall score wins, a gap under 0.1 is a draw;
and **field** (K = 8), where every opinion plays a fixed "average durian" rated 1500 and scores its
own overall score - that is how the ratings feed in. The gold Elo uses the writer's own score where
there is one, else the gold sentiment (positive 1, mixed or neutral 0.5, negative 0); each
solution's Elo uses its own `overall_rating`, `compares` and `preferred`. The named opponent comes
from an alias match on the quote, the same for everyone.

In [ ]:
gold_rows = elo.gold_rows(opinions)
solution_rows = {
    solution.label: elo.solution_rows(opinions, results[results["solution"] == solution.id])
    for solution in to_run
}
gold_elo = elo.durian_elo(gold_rows)
solution_elos = {label: elo.durian_elo(rows) for label, rows in solution_rows.items()}
elo_table = elo.elo_table(gold_elo, solution_elos)
print("gold matches:", gold_elo.matches, "- the people's king:", gold_elo.king)
elo_table

Besides the Elo, the plain numbers: the mean overall rating each solution gave each durian, next to
the people's score (the writer's own score where there is one, else the sentiment mapped to 1 /
0.5 / 0 - the same score the gold Elo plays with).

In [ ]:
ratings = elo.ratings_by_durian(gold_rows, solution_rows)
ratings.round(2)

In [ ]:
elo_figure = charts.durian_elo(
    gold_elo.history,
    {label: result.rating for label, result in solution_elos.items()},
    chart_dir / "06_durian_elo.png",
    run_date,
)
elo_figure.show()

## 7. Wrap-up

A full run saves `results/answers.csv`, `metrics.csv`, `elo.csv`, `ratings_by_durian.csv`, the two
confidence tables and `summary.md`, and rewrites the results block of `README.md` with a Singapore
timestamp: the main table, the confidence tables, links to the charts and the observations from
`results/observations.md` (hand-written; the notebook generates sentences when that file is
missing). The Elo and the ratings per durian stay in `summary.md`. A smoke run saves the same files
under `results/smoke/` and leaves the README alone.

In [ ]:
results_dir = ROOT / "results" / "smoke" if SMOKE else ROOT / "results"
results_dir.mkdir(parents=True, exist_ok=True)
results.to_csv(results_dir / "answers.csv", index=False)
summary.to_csv(results_dir / "metrics.csv")
elo_table.to_csv(results_dir / "elo.csv")
ratings.to_csv(results_dir / "ratings_by_durian.csv")
confidence_table.to_csv(results_dir / "confidence_by_bin.csv")
cutoff_display.to_csv(results_dir / "confidence_cutoffs.csv")

info = report.RunInfo(
    n_opinions=len(opinions),
    mode="smoke" if SMOKE else "full",
    timestamp=report.now_singapore(),
    skipped=benchmarking.skipped(solutions),
    machine=machine,
)
chart_paths = sorted(chart_dir.glob("*.png"))
observations_path = ROOT / "results" / "observations.md"
notes = report.read_observations(observations_path) if observations_path.exists() else None
if SMOKE:
    summary_path = results_dir / "summary.md"
    summary_path.write_text(
        report.summary_markdown(
            summary,
            elo_table,
            info,
            ratings=ratings,
            confidence=confidence_table,
            cutoffs=cutoff_display,
            notes=notes,
        ),
        encoding="utf-8",
    )
    print(f"smoke run: wrote {summary_path.relative_to(ROOT)}; README untouched")
else:
    report.write_results(
        ROOT / "README.md",
        results_dir / "summary.md",
        summary,
        elo_table,
        info,
        chart_paths,
        main_only,
        ratings,
        confidence=confidence_table,
        cutoffs=cutoff_display,
        notes=notes,
    )
    print(f"full run: wrote results/summary.md and the README results block ({info.timestamp})")
for path in chart_paths:
    width, height = charts.png_size(path)
    print(f"{path.relative_to(ROOT)}: {width} x {height} px")

## 8. Bonus: the same questions through Pydantic AI

Pydantic AI (2.60, October 2026) treats a decision model like any other model: `typesafe:jev-latest`,
`system-one:<model>` (Ollama) and `openai-decisions:gpt-6-luna` sit next to `openai:gpt-6-luna`.
Each field of the agent's `output_type` becomes one question - a `bool` is a yes/no, a `Literal` a
pick-one, whole numbers with a description per level a score - all sent in one request, and the
probabilities come back in the response's `provider_details`. The benchmark above calls the APIs
directly so you can see the payloads; this is what the same questions look like inside an agent
framework, the shape in which decision models will do routing and pre-filtering in agentic apps.

In [ ]:
from typing import Literal

from pydantic import BaseModel, Field
from pydantic_ai import Agent
from pydantic_ai.models.system_one import SystemOneModel
from pydantic_ai.providers.system_one import SystemOneProvider

question_text = {question.name: question.instructions for question in QUESTIONS}


class DurianVerdict(BaseModel):
    """What one opinion about a durian says."""

    taste_mentioned: bool = Field(description=question_text["taste_mentioned"])
    compares: bool = Field(description=question_text["compares"])
    overall_sentiment: Literal["positive", "mixed", "negative", "neutral"] = Field(
        description=question_text["overall_sentiment"]
    )
    variety: Literal["musang_king", "black_thorn", "red_prawn", "not_named"] = Field(
        description=question_text["variety"]
    )


backends = {
    "jev": "typesafe:jev-latest",
    "openai_decisions": "openai-decisions:gpt-6-luna",
    "kev": SystemOneModel(
        "kev:4b-q8_0", provider=SystemOneProvider(base_url=benchmarking.OLLAMA_HOST)
    ),
    "ollama_clef_flash": SystemOneModel(
        "clef-flash", provider=SystemOneProvider(base_url=benchmarking.OLLAMA_HOST)
    ),
    "gpt_6_luna": "openai:gpt-6-luna",
}
for solution_id, model in backends.items():
    if ready(solution_id):
        agent = Agent(
            model, output_type=DurianVerdict, instructions="Read one opinion about a durian."
        )
        result = await agent.run(example.quote)
        print(f"\n{solution_id}: {result.output}")
        print(f"{solution_id}: {result.response.provider_details}")  # the probabilities
        print(f"{solution_id}: {result.usage}")